# Observability and benchmarking

## Learning goals
- Define RTF, JCT, TTFT and compute them on simulated data
- Avoid the classic benchmarking pitfalls
- Read the paper's headline speedups as targets, not magic

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## The metrics that matter

- **TTFT** — time to first token/output; dominates *perceived* latency.
- **JCT** — job completion time; the full request end to end.
- **RTF** — real-time factor for audio: generation_time / audio_duration. RTF < 1 means faster-than-real-time (good for live speech).

We compute all three on a simulated request set.

In [ ]:
requests = [
    {'ttft': 0.30, 'jct': 4.2, 'audio_s': 6.0},
    {'ttft': 0.41, 'jct': 5.1, 'audio_s': 5.5},
    {'ttft': 0.28, 'jct': 3.9, 'audio_s': 7.2},
]
def rtf(r): return r['jct'] / r['audio_s']
mean = lambda xs: sum(xs) / len(xs)
print(f"mean TTFT {mean([r['ttft'] for r in requests]):.2f}s")
print(f"mean JCT  {mean([r['jct'] for r in requests]):.2f}s")
print(f"mean RTF  {mean([rtf(r) for r in requests]):.2f}  (<1 is real-time)")

## Pitfalls that make you lie to yourself

1. **Reporting the mean, hiding the tail.** p99 latency is what users feel. Plot the distribution.
2. **Warm vs cold.** The first request pays compilation/allocation costs; drop warmup runs.
3. **Batch size of one.** Serving throughput comes from batching; a single-request benchmark measures the wrong thing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
rng = np.random.default_rng(3)
# a realistic latency sample: mostly fast, with a heavy tail
latencies = np.concatenate([rng.normal(0.4, 0.05, 480), rng.normal(1.2, 0.3, 20)])
p50, p99 = np.percentile(latencies, [50, 99])
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(latencies, bins=40, color='#4c78a8')
ax.axvline(p50, color='green', label=f'p50={p50:.2f}s')
ax.axvline(p99, color='red', label=f'p99={p99:.2f}s')
ax.set_xlabel('latency (s)'); ax.set_ylabel('requests')
ax.set_title('Mean hides the tail'); ax.legend()
fig.tight_layout(); plt.show()
print(f'mean={latencies.mean():.2f}s but p99={p99:.2f}s')

## Targets from the paper

Disaggregated vLLM-Omni reports large wins over a monolithic baseline. Treat these as what *good* looks like when the bottleneck stage is properly scaled.

In [ ]:
targets = {
    'Qwen3-Omni': 'RTF -90.7%, JCT -91.4%, Thinker TPS 12.97x, Talker TPS 7.98x',
    'BAGEL':      'T2I 2.40x, I2I 3.72x',
    'MiMo-Audio': 'RTF 1.39 -> 0.12 with graph compilation (11.58x)',
}
for model, wins in targets.items():
    print(f'{model:12}: {wins}')

## Exercise

Given the latency sample above, would reporting only the mean overstate or understate the user experience? Compute the gap.

In [ ]:
# Solution
print(f'p99 is {p99/latencies.mean():.1f}x the mean -- reporting the mean understates tail pain.')

## Source lab

Run the repo's `benchmarks/` against a served model on Linux. Compare your RTF and JCT to the targets above and see which stage limits you.